# 04 Data Preprocessing

## Objective

The objective of this notebook is to prepare the dataset for machine learning by applying appropriate preprocessing techniques while preventing data leakage.

This notebook focuses on:

- Reviewing data quality
- Handling duplicate records
- Preparing feature and target variables
- Applying feature scaling where necessary
- Splitting the dataset into training and testing sets
- Addressing severe class imbalance
- Saving processed datasets for model training

## 2. Import Libraries

Import necessary Python packages for data handling, preprocessing, feature scaling, resampling, and visualization.

In [1]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# Machine Learning - Data Preprocessing
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler

# ==========================================
# Handling Imbalanced Data
# ==========================================
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

# ==========================================
# Utility Libraries
# ==========================================
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# Display Settings
# ==========================================
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
import joblib
import warnings

# Display configurations
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")


All libraries imported successfully.


## 3. Load Dataset

Load the raw dataset from `data/raw/creditcard.csv`.

In [7]:
# Load dataset — raw string to avoid backslash escape issues
DATA_PATH = Path(r"D:\Fraud Detection\data\raw\creditcard.csv")

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Dataset loaded: 284,807 rows x 31 columns


## 4. Initial Data Quality Review

### Objective

Before applying any preprocessing techniques, verify the current quality and integrity of the dataset.

This review confirms that the dataset matches the findings from the Exploratory Data Analysis (EDA) phase and ensures that preprocessing decisions are based on accurate and consistent data.

### Why is this review important?

Data preprocessing should never begin without verifying the current state of the dataset.

This review ensures that:

- The correct dataset has been loaded.
- No unexpected changes have occurred since the EDA phase.
- Data quality issues are clearly identified before preprocessing.

In [8]:
df.shape

(284807, 31)

In [9]:
df.dtypes

Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

In [10]:
df.isnull().sum()

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

In [11]:
df.duplicated().sum()

np.int64(1081)

In [12]:
df["Class"].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

### Conclusion

The initial data quality review confirms that the dataset remains consistent with the findings from the EDA phase. No missing values are present, and all variables are stored in the expected format. However, duplicate records and severe class imbalance require careful consideration before model training. These issues will be addressed systematically in the following preprocessing steps.

## 5. Duplicate Handling

### Objective

The objective of this section is to investigate duplicate records in the dataset and determine whether they should be retained or removed before model training.

Rather than treating duplicates as automatic data quality issues, this analysis evaluates their potential business significance and impact on machine learning performance.

### Business Questions

1. Are duplicate records always data quality issues?

2. Could duplicate transactions occur in real banking systems?

3. What risks are associated with removing duplicate records?

4. What risks are associated with keeping duplicate records?

5. Based on the available data, should duplicate records be removed?

### Why is this analysis important?

Duplicate records can either represent data quality problems or legitimate repeated transactions.

Removing duplicates without investigation may discard valuable fraud patterns, while retaining unnecessary duplicates may bias machine learning models.

Therefore, duplicate records must be evaluated carefully before any preprocessing decisions are made.

In [13]:
df.duplicated().sum()

np.int64(1081)

In [14]:
class_counts = df['Class'].value_counts()
class_pcts = df['Class'].value_counts(normalize=True) * 100

target_dist = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_pcts
})

target_dist.index = ['Legitimate (0)', 'Fraudulent (1)']
target_dist

,Count,Percentage (%)
Legitimate (0),284315,99.8273
Fraudulent (1),492,0.1727


In [19]:
duplicates = df[df.duplicated(keep=False)]

# Count duplicate records by class
duplicate_summary = (
    duplicates.groupby("Class")
              .size()
              .reset_index(name="Duplicate Count")
)

# Rename class labels
duplicate_summary["Class"] = duplicate_summary["Class"].map({
    0: "Legitimate",
    1: "Fraud"
})

duplicate_summary

,Class,Duplicate Count
0,Legitimate,1822
1,Fraud,32


In [20]:
duplicate_summary["Percentage"] = (
    duplicate_summary["Duplicate Count"]
    / duplicate_summary["Duplicate Count"].sum()
    * 100
).round(2)

duplicate_summary

,Class,Duplicate Count,Percentage
0,Legitimate,1822,98.2700
1,Fraud,32,1.7300


In [21]:
if duplicates["Class"].nunique() == 2:
    print("Duplicates are present in both legitimate and fraudulent transactions.")
elif duplicates["Class"].iloc[0] == 0:
    print("Duplicates are present only in legitimate transactions.")
else:
    print("Duplicates are present only in fraudulent transactions.")

Duplicates are present in both legitimate and fraudulent transactions.


In [22]:
df = df.drop_duplicates().reset_index(drop=True)

In [23]:
df.duplicated().sum()

np.int64(0)

In [25]:
df.shape

(283726, 31)

In [24]:
# Save the cleaned dataset
df.to_csv(
    "../data/processed/creditcard_no_duplicates.csv",
    index=False
)

### Observation

The dataset contained 1,081 exact duplicate records. These duplicates were identical across all 31 columns and therefore did not contribute additional information for supervised learning.

To prevent repeated observations from influencing the model, the duplicate records were removed while preserving the original dataset.

### Decision

The exact duplicate records were removed because:

- They do not provide additional information.
- They may bias the learning algorithm by giving more weight to repeated observations.
- Removing them improves data quality while maintaining the integrity of the original dataset.

The original raw dataset has been preserved for reproducibility, and the cleaned dataset has been saved separately for further preprocessing.

## 6. Missing Value Handling

### Objective

The objective of this section is to identify missing values in the dataset and determine whether any preprocessing techniques are required to handle incomplete data before model training.

### Why is this step important?

Missing values can reduce model performance, introduce bias, or cause certain machine learning algorithms to fail. Therefore, every dataset should be examined for missing values before proceeding with feature engineering and model training.

In [27]:
df.isnull().sum()

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

## 7. Feature & Target Separation

### Objective

The objective of this section is to separate the predictor variables (features) from the target variable before splitting the dataset into training and testing sets.

This ensures that the machine learning models receive only the input features during training while the target variable is used for supervised learning.

### Why is this step important?

Machine learning models learn relationships between input features and a target variable.

Separating the target before preprocessing ensures:

- Clear distinction between predictors and the outcome.
- Easier preprocessing of feature variables.
- Prevention of accidental modification of the target variable.

In [28]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [30]:
X.shape

(283726, 30)

In [32]:
y.shape

(283726,)

## 8. Train-Test Split

### Objective

The objective of this section is to divide the dataset into training and testing subsets. This ensures that the machine learning model is trained on one portion of the data and evaluated on previously unseen data, providing an unbiased estimate of model performance.

### Why is this step important?

Machine learning models should be evaluated on data they have never seen during training.

Separating the dataset into training and testing sets helps:

- Measure how well the model generalizes to new transactions.
- Prevent overfitting.
- Provide a realistic estimate of real-world performance.

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [34]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(226980, 30)
(56746, 30)
(226980,)
(56746,)


In [35]:
train_distribution = (
    y_train.value_counts(normalize=True)
           .mul(100)
           .round(2)
           .rename("Percentage")
)

test_distribution = (
    y_test.value_counts(normalize=True)
          .mul(100)
          .round(2)
          .rename("Percentage")
)

print("Training Set")
print(train_distribution)

print("\nTesting Set")
print(test_distribution)

Training Set
Class
0   99.8300
1    0.1700
Name: Percentage, dtype: float64

Testing Set
Class
0   99.8300
1    0.1700
Name: Percentage, dtype: float64


## 9. Feature Scaling

### Objective

The objective of this section is to scale numerical features so that variables with larger numerical ranges do not dominate the learning process. Scaling is performed after the train-test split to prevent data leakage and ensure a fair evaluation of machine learning models.

### Why is this step important?

Many machine learning algorithms are sensitive to differences in feature scales. Features with large numerical values can disproportionately influence model training.

Scaling transforms numerical features into comparable ranges, allowing algorithms to learn more effectively.

In [36]:
# Features to scale
features_to_scale = ["Time", "Amount"]

# Initialize scaler
scaler = RobustScaler()

# Fit only on training data
X_train[features_to_scale] = scaler.fit_transform(
    X_train[features_to_scale]
)

# Transform test data using the same scaler
X_test[features_to_scale] = scaler.transform(
    X_test[features_to_scale]
)

In [37]:
X_train

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
225399,0.7017,2.2390,-1.7245,-2.1515,-2.5778,0.9937,3.5655,-1.7860,0.8601,-1.2640,1.5679,-0.2357,-0.7056,0.3635,-0.3135,0.2850,-0.5776,0.3815,-0.1808,-0.3181,-0.3238,-0.1496,-0.0493,0.2784,0.6847,-0.2190,-0.1592,0.0379,-0.0499,0.1375
133746,-0.0482,-1.3151,1.6308,0.5970,-0.0384,-0.4046,-0.9657,0.2122,0.7354,-1.2679,-0.4826,1.4371,1.7624,1.2543,0.8627,-0.2662,0.3938,-0.2653,-0.3436,0.3924,-0.0676,-0.2389,-0.9468,0.3239,0.5156,-0.7130,-0.2665,-0.0178,0.0511,-0.2091
185792,0.4969,1.9088,0.0212,-2.0880,0.1293,1.1615,0.6052,-0.0224,0.1803,0.2838,-0.4978,1.5330,1.0392,0.4757,-0.6895,0.9641,-0.2775,0.7933,-0.1567,-0.9947,-0.2105,0.2936,1.0958,-0.0449,-1.6895,0.1061,0.0078,0.0452,-0.0531,-0.0988
148925,0.0767,1.8113,0.3166,0.3168,3.8802,0.0485,1.0202,-0.7349,0.2337,0.6814,1.1467,1.3373,-1.7751,2.1788,1.2396,-2.0371,1.3441,-0.4752,0.8248,-1.6935,-0.2280,0.1389,0.7004,0.1741,0.7030,-0.2125,-0.0100,-0.0177,-0.0380,-0.0662
18398,-0.6496,1.3588,-1.1209,0.5503,-1.5477,-1.1950,0.2754,-1.2018,0.2129,-2.0943,1.4928,1.7378,0.0144,0.3298,-0.0446,0.4731,-0.6597,0.7875,-0.6414,-0.5918,-0.3617,-0.3410,-0.6364,0.2528,-0.3442,-0.0643,-0.4396,0.0625,0.0131,0.0266
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224284,0.6966,-0.0983,-0.3828,0.2027,-0.7323,0.3363,-0.2539,0.8379,-0.4026,-1.4245,0.6654,0.2895,0.4168,1.0224,0.0500,-0.1822,-2.0298,-0.4403,2.2415,0.4504,0.0288,0.1548,1.0093,0.0021,-0.4629,-0.4225,0.0106,0.0411,-0.0045,1.4124
3493,-0.9610,-1.7089,-1.2963,1.9532,-2.2770,-0.5104,0.4088,0.3292,0.4773,1.0581,-2.0158,1.3879,0.8179,-0.9362,0.0831,0.6829,-0.5921,0.0558,0.0950,-0.3048,0.5574,0.4668,0.8609,0.3613,-0.2647,0.4235,0.0657,-0.0264,0.0776,3.3221
241772,0.7845,-0.8754,1.1600,0.5740,1.1789,-0.2063,0.2606,0.3711,0.4435,-0.0116,-0.1927,0.0196,0.8761,-0.9089,0.2730,-2.1673,-1.0683,0.5000,-0.1369,1.5073,-0.3539,-0.1006,-0.1577,0.0498,-0.0142,-0.5139,-0.7720,-0.2430,0.0949,0.2068
60347,-0.4161,1.2470,0.3486,0.5894,0.9830,-0.2220,-0.4527,-0.0123,-0.1402,0.0522,-0.0694,-0.5863,0.7145,1.3144,-0.0283,1.0801,0.4654,-0.7949,-0.1707,-0.1880,-0.0428,-0.2115,-0.5560,0.0371,-0.1333,0.4036,-0.5958,0.0439,0.0320,-0.2367


In [38]:
X_test

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
86249,-0.2764,1.2288,-0.0634,0.2741,0.6475,-0.0481,0.3721,-0.2242,0.0799,0.6408,-0.2731,-1.2527,0.4651,0.4005,-0.2928,-0.1018,-0.3998,0.0343,-0.7836,0.1413,-0.0966,-0.1296,-0.0838,-0.1517,-0.7004,0.5986,0.4914,0.0030,0.0018,-0.1466
250634,0.8291,-0.2032,1.1767,-0.7596,-0.5185,0.6296,-0.7217,0.6389,0.2434,-0.1575,-0.5066,-1.3948,-0.0925,0.0216,0.7308,0.4963,-0.1996,-0.4344,0.3487,0.1625,-0.0984,0.3846,1.2068,-0.0828,0.5084,-0.7109,-0.2345,0.3796,0.2614,-0.2791
20163,-0.6336,-1.6728,1.4013,1.5039,2.1755,0.6998,1.0621,1.1144,-0.5358,-0.2530,3.0712,0.7686,0.2021,1.3492,-0.9506,0.9220,0.3670,-1.2155,0.5986,1.4260,0.8285,-0.5259,-0.4087,-0.2801,-0.8465,-0.1555,-0.0624,0.0078,0.1139,0.9590
68688,-0.3714,0.8194,-1.1249,0.5150,0.5139,-1.0090,0.4885,-0.5807,0.1877,-0.9991,0.8720,0.9066,0.3864,0.1015,0.2755,0.7035,-1.0050,-0.4259,1.7740,-1.3996,-0.1101,-0.0572,-0.1681,-0.1987,-0.3374,0.2385,-0.2895,0.0382,0.0584,2.8537
191151,0.5241,2.0097,0.1056,-1.7528,0.5883,0.3748,-0.6379,0.0093,-0.1295,0.4926,-0.5270,0.9851,1.0365,0.4176,-1.6419,-1.1235,0.3168,0.8920,0.6106,0.2158,-0.1355,0.0090,0.3574,-0.0135,-0.4469,0.1115,0.6429,-0.0370,-0.0434,-0.2864
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56277,-0.4388,-1.1625,0.7957,1.9534,1.6542,1.0690,-0.1835,0.0932,0.3776,-1.3622,-0.3982,1.6973,0.3106,0.0036,-1.1949,-0.1689,1.3266,0.1570,0.7299,-2.0175,0.0239,0.2585,0.5022,-0.1219,0.0949,-0.0790,-0.0846,0.0931,0.1462,-0.2783
187751,0.5070,-2.0233,2.2143,-2.0975,-1.0419,-0.1209,-0.0277,-0.5952,1.8210,-0.3916,-0.2335,-0.6720,1.2845,0.7313,1.2189,-1.1203,0.8645,-0.4483,0.1971,0.5083,0.0150,-0.2023,-0.8106,0.1652,-0.3600,-0.1037,0.1731,0.0607,-0.0034,-0.1675
115100,-0.1281,1.1807,-0.1992,1.2336,0.8692,-1.1388,-0.3159,-0.6608,0.1036,1.0116,-0.2246,-0.8232,0.1567,-0.4898,-0.3007,0.2585,0.1879,-0.1437,-0.1703,-0.0370,-0.1343,-0.0978,-0.1141,0.0379,0.4195,0.2363,0.3181,0.0134,0.0262,-0.1793
186966,0.5029,2.0611,0.1367,-1.8308,0.2064,0.7051,-0.3262,0.1073,-0.0933,0.1847,-0.2231,0.8747,0.9758,0.5477,-0.7707,-0.5224,0.6006,0.0552,0.2506,0.4832,-0.1022,-0.3328,-0.8610,0.2749,0.1133,-0.2200,0.1763,-0.0636,-0.0436,-0.2937


### Observation

The `Time` and `Amount` features were successfully scaled using RobustScaler. The scaler was fitted only on the training dataset and then applied to both training and testing datasets, preventing data leakage.

### Decision

Only the `Time` and `Amount` features were scaled because the PCA-transformed variables (`V1–V28`) are already standardized. RobustScaler was selected due to the presence of significant outliers in the transaction amount feature.

## 10. Handling Class Imbalance

### Objective

The objective of this section is to address the severe class imbalance in the training dataset by evaluating different resampling techniques. These methods aim to improve the model's ability to identify fraudulent transactions while maintaining reliable performance on legitimate transactions.

Resampling techniques will be applied only to the training dataset to prevent data leakage and preserve the integrity of model evaluation.

### Why is this step important?

Fraudulent transactions represent only a very small fraction of all transactions. A machine learning model trained on such an imbalanced dataset may become biased toward predicting the majority class, resulting in poor fraud detection.

Handling class imbalance helps the model learn patterns from both legitimate and fraudulent transactions more effectively.

## 11. Preprocessing Summary

Summarize preprocessing results, transformations, sample sizes, and class balance distributions.

## 12. Save Processed Data

Export processed feature and target datasets to `data/processed/`.

## 13. Next Steps

Outline next steps for baseline model evaluation in `05_Baseline_Models.ipynb`.